In [1]:
# Устанавливаем библиотеки "партизана"
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q -U duckduckgo-search langchain langchain-community
print("Арсенал загружен. Мы готовы.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 85.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Наш выбор: Qwen 2.5 (1.5 млрд параметров) - легкая и злая
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"🚀 Начинаю загрузку нейросети {model_id}...")

# 1. Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Загружаем саму модель
# device_map="auto" - автоматически использует твои 2 видеокарты T4
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

print("✅ Нейросеть загружена в память GPU. Ждет команд.")

🚀 Начинаю загрузку нейросети Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Нейросеть загружена в память GPU. Ждет команд.


In [3]:
from duckduckgo_search import DDGS

# --- Функция 1: Глаза (Поиск в интернете) ---
def search_web(query, max_results=3):
    print(f"🌐 Ищу свежие данные: '{query}'...")
    try:
        # Используем DuckDuckGo для анонимного поиска
        results = DDGS().text(query, max_results=max_results)
        context = ""
        if results:
            for res in results:
                context += f"ИСТОЧНИК: {res['title']}\nТЕКСТ: {res['body']}\n\n"
        else:
            context = "Ничего не найдено."
        return context
    except Exception as e:
        return f"Ошибка поиска: {e}"

# --- Функция 2: Мозг (Генерация ответа) ---
def symbiont_agent(user_query):
    # 1. Сначала идем в интернет
    web_data = search_web(user_query)
    
    # 2. Формируем инструкцию для ИИ (System Prompt)
    # Мы приказываем ей отвечать только на основе найденного
    prompt = f"""
    <|im_start|>system
    Ты — аналитический ИИ-агент. Твоя задача — ответить на вопрос пользователя, используя ТОЛЬКО предоставленные ниже результаты поиска.
    Не выдумывай факты. Если информации нет в поиске, так и скажи.
    Отвечай емко и на русском языке.
    
    РЕЗУЛЬТАТЫ ПОИСКА:
    {web_data}
    <|im_end|>
    <|im_start|>user
    {user_query}
    <|im_end|>
    <|im_start|>assistant
    """
    
    # 3. Превращаем текст в токены
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 4. Генерируем мысль
    outputs = model.generate(
        **inputs, 
        max_new_tokens=512,  # Максимальная длина ответа
        temperature=0.6      # Баланс между точностью и креативом
    )
    
    # 5. Декодируем ответ
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Очищаем технические теги, оставляем только суть
    # (Ищем последнее вхождение assistant и берем текст после него)
    if "<|im_start|>assistant" in response:
        final_answer = response.split("<|im_start|>assistant")[-1].strip()
    else:
        final_answer = response # На случай сбоя форматирования
        
    return final_answer

print("✅ Агент 'Симбионт' активирован. Система готова к бою.")

✅ Агент 'Симбионт' активирован. Система готова к бою.


In [6]:
import warnings
# 1. Глушим системные крики (игнорируем предупреждения)
warnings.filterwarnings('ignore')

print("🔇 Системный шум заглушен.")

import yfinance as yf
# Проверяем, встала ли библиотека
try:
    test = yf.Ticker("AAPL")
    print("✅ Финансовый модуль (yfinance) АКТИВЕН.")
except Exception as e:
    print("⚠️ Модуль не найден, пробую установить тихо...")
    !pip install -q yfinance
    import yfinance as yf
    print("✅ Теперь точно работает.")

# --- КОД АГЕНТА ---

def market_analyst(ticker_symbol):
    print(f"\n🕵️‍♂️ Агент начинает расследование по: {ticker_symbol}...")
    
    try:
        # Получаем данные с биржи
        stock = yf.Ticker(ticker_symbol)
        
        # Цена
        hist = stock.history(period="1d")
        if not hist.empty:
            current_price = round(hist['Close'].iloc[-1], 2)
        else:
            current_price = "Нет данных (рынок закрыт?)"
            
        # Новости
        news = stock.news
        news_summary = ""
        if news:
            print(f"📰 Найдено {len(news)} свежих новостей.")
            for i, item in enumerate(news[:3]):
                news_summary += f"{i+1}. {item['title']} (Ссылка: {item['link']})\n"
        else:
            news_summary = "Новостей нет."
            
        report = f"""
        === ОТЧЕТ ПО {ticker_symbol} ===
        💰 Цена сейчас: {current_price} USD
        
        🗞 Главные заголовки:
        {news_summary}
        =============================
        """
        return report

    except Exception as e:
        return f"❌ Сбой подключения: {e}"

# --- ТЕСТ ---
# Проверяем на Nvidia (NVDA)
print(market_analyst("NVDA"))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔇 Системный шум заглушен.
✅ Финансовый модуль (yfinance) АКТИВЕН.

🕵️‍♂️ Агент начинает расследование по: NVDA...
📰 Найдено 10 свежих новостей.
❌ Сбой подключения: 'title'


In [7]:
import yfinance as yf

def safe_market_analyst(ticker_symbol):
    print(f"\n🕵️‍♂️ Попытка №2. Агент сканирует: {ticker_symbol}...")
    
    try:
        stock = yf.Ticker(ticker_symbol)
        
        # 1. Цена (тут обычно без сбоев)
        try:
            hist = stock.history(period="1d")
            if not hist.empty:
                current_price = round(hist['Close'].iloc[-1], 2)
            else:
                current_price = "Рынок закрыт/Нет данных"
        except:
            current_price = "Ошибка получения цены"

        # 2. Новости (Самое сложное место)
        news = stock.news
        news_summary = ""
        
        if news:
            print(f"📰 Найдено {len(news)} записей. Анализирую структуру...")
            
            # Берем первые 3 новости
            for i, item in enumerate(news[:3]):
                # Ищем заголовок под разными именами (защита от смены формата)
                title = item.get('title') or item.get('headline') or "Заголовок скрыт"
                link = item.get('link') or item.get('url') or "Нет ссылки"
                
                news_summary += f"{i+1}. {title}\n   Ссылка: {link}\n"
        else:
            news_summary = "Новостей не найдено."

        # 3. Формируем отчет
        report = f"""
        === СЕКРЕТНЫЙ ОТЧЕТ: {ticker_symbol} ===
        💰 Цена: {current_price} USD
        
        🗞 Сводка:
        {news_summary}
        =======================================
        """
        return report

    except Exception as e:
        return f"❌ Полный сбой: {e}"

# ТЕСТ
print(safe_market_analyst("NVDA"))


🕵️‍♂️ Попытка №2. Агент сканирует: NVDA...
📰 Найдено 10 записей. Анализирую структуру...

        === СЕКРЕТНЫЙ ОТЧЕТ: NVDA ===
        💰 Цена: 177.19 USD
        
        🗞 Сводка:
        1. Заголовок скрыт
   Ссылка: Нет ссылки
2. Заголовок скрыт
   Ссылка: Нет ссылки
3. Заголовок скрыт
   Ссылка: Нет ссылки

        


In [8]:
!pip install -q wikipedia
print("✅ Модуль энциклопедических знаний подключен.")

  Preparing metadata (setup.py) ... done
✅ Модуль энциклопедических знаний подключен.


In [9]:
import yfinance as yf
import wikipedia

# Настраиваем Википедию на русский язык
wikipedia.set_lang("ru")

def fusion_analyst(ticker, company_name):
    print(f"🔄 Запуск протокола слияния данных для {company_name}...")
    
    # 1. ПОЛУЧАЕМ ЦЕНУ (Yahoo)
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="1d")
        price = round(data['Close'].iloc[-1], 2)
        financial_status = f"Актуальная цена акции {ticker}: ${price}"
    except:
        financial_status = "Данные с биржи временно недоступны."

    # 2. ПОЛУЧАЕМ ЗНАНИЯ (Wikipedia)
    try:
        # Ищем страницу и берем краткое содержание (5 предложений)
        wiki_text = wikipedia.summary(company_name, sentences=5)
    except:
        wiki_text = "Статья в энциклопедии не найдена."

    # 3. НЕЙРОСЕТЬ АНАЛИЗИРУЕТ (Qwen)
    prompt = f"""
    <|im_start|>system
    Ты — финансовый консультант. Твоя задача — объединить биржевые данные и справочную информацию, чтобы дать краткую справку о компании.
    
    ВХОДНЫЕ ДАННЫЕ:
    1. {financial_status}
    2. Справка: {wiki_text}
    
    Напиши вывод: что это за компания и сколько она сейчас стоит.
    <|im_end|>
    <|im_start|>user
    Расскажи мне про {company_name}
    <|im_end|>
    <|im_start|>assistant
    """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return response.split("<|im_start|>assistant")[-1].strip()

# --- ТЕСТ ---
# Спрашиваем про NVIDIA
print(fusion_analyst("NVDA", "Nvidia"))

🔄 Запуск протокола слияния данных для Nvidia...
system
    Ты — финансовый консультант. Твоя задача — объединить биржевые данные и справочную информацию, чтобы дать краткую справку о компании.
    
    ВХОДНЫЕ ДАННЫЕ:
    1. Актуальная цена акции NVDA: $177.19
    2. Справка: Nvidia (/ɛnˈvɪdiə/; NVIDIA Corporation) — американская технологическая компания, разработчик графических процессоров и систем на чипе (SoC). Разработки компании получили распространение в индустрии видеоигр, сфере профессиональной визуализации, области высокопроизводительных вычислений (в том числе ИИ) и автомобильной промышленности, где бортовые компьютеры Nvidia используются в качестве основы для беспилотных автомобилей.
Компания была основана в 1993 году. На IV квартал 2018 года была крупнейшим в мире производителем компьютерной-совместимой дискретной графики с долей 81,2 % (статистика включает все графические процессоры, доступные для прямой покупки конечными пользователями — GeForce, Quadro и ускорители вычис